In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import shutil
import re
import seaborn as sns
from itertools import combinations
import numpy as np
import xgboost
import lightgbm
import tensorflow as tf
from pymatgen.core.structure import Structure
import ast

2025-09-19 21:29:10.447876: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-19 21:29:10.560206: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-19 21:29:10.563156: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-09-19 21:29:10.563170: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore 

In [2]:
pd.set_option('display.max_rows', 2000)
pd.set_option('display.max_columns', 2000)
pd.set_option('display.width', 2000)
pd.set_option('display.max_colwidth', 2000)

In [3]:
df_raw_N2 = pd.read_csv(f'./dataframes/raw_df_N2_bandcenter_expanded_v5.csv')
df_raw_H2 = pd.read_csv(f'./dataframes/raw_df_H2_bandcenter_expanded_v5.csv')
df_raw_N2H = pd.read_csv(f'./dataframes/raw_df_N2H_bandcenter_expanded_v5.csv')
df_raw_NH3 = pd.read_csv(f'./dataframes/raw_df_NH3_bandcenter_expanded_v5.csv')

In [4]:
df_N2 = df_raw_N2.copy()
df_H2 = df_raw_H2.copy()
df_N2H = df_raw_N2H.copy()
df_NH3 = df_raw_NH3.copy()

In [5]:
df_NH3.loc[20,'mol_output_all_coords']

"[('N', array([ 5.01085368,  6.0782592 , 10.01142383])), ('H', array([ 4.1047193 ,  6.32094312, 10.43149707])), ('H', array([ 5.74415857,  6.64829608, 10.44330322])), ('H', array([ 5.18949003,  5.08648036, 10.23367231]))]"

In [6]:
dfs_raw = [df_raw_N2,
      df_raw_H2,
      df_raw_N2H,
      df_raw_NH3]
mol_names = ['N2',
            'H2',
            'N2H',
            'NH3']
for i in range(len(dfs_raw)):
    each_df = dfs_raw[i]
    name = mol_names[i]
    print(name,each_df.shape)

N2 (489, 637)
H2 (517, 637)
N2H (463, 787)
NH3 (525, 937)


In [7]:
df_N2['n0_n11_dos'].dropna().apply(ast.literal_eval).apply(lambda d: 'error' in d if isinstance(d, dict) else False).sum()

10

In [8]:
df = df_N2
mask = df['n0_n11_dos'].dropna().apply(
    lambda x: 'error' in ast.literal_eval(x) if isinstance(x, str) else False
)

rows_with_error = df.loc[mask.index[mask]]
rows_with_error['n0_n11_dos']

140     {'error': 'no element found: line 983, column 0'}
141     {'error': 'no element found: line 983, column 0'}
142     {'error': 'no element found: line 983, column 0'}
143     {'error': 'no element found: line 983, column 0'}
144     {'error': 'no element found: line 983, column 0'}
145     {'error': 'no element found: line 983, column 0'}
146     {'error': 'no element found: line 983, column 0'}
147     {'error': 'no element found: line 983, column 0'}
251    {'error': 'no element found: line 1745, column 0'}
252    {'error': 'no element found: line 1745, column 0'}
Name: n0_n11_dos, dtype: object

In [10]:
def flatten_dos_columns(df):
    """Flatten 'n0_n{i}_dos' dictionary columns into separate feature columns"""
    dos_cols = [col for col in df.columns if re.match(r"n0_n\d+_dos", col)]
    flat_records = []

    for idx, row in df.iterrows():
        flat_row = {}
        for col in dos_cols:
            entry = row.get(col, {})
            try:
                parsed = ast.literal_eval(entry) if isinstance(entry, str) else entry
            except:
                parsed = {}

            if isinstance(parsed, dict):
                for k, v in parsed.items():
                    flat_col = f"{col}_{k}"
                    # If it's a list, convert to string or take first
                    flat_row[flat_col] = v[0] if isinstance(v, list) and v else v
        flat_records.append(flat_row)

    flat_df = pd.DataFrame(flat_records).fillna(0.0)
    df_out = pd.concat([df.drop(columns=dos_cols).reset_index(drop=True), flat_df], axis=1)
    return df_out

In [11]:
df_N2 = flatten_dos_columns(df_N2)
df_N2H = flatten_dos_columns(df_N2H)
df_NH3 = flatten_dos_columns(df_NH3)
df_H2 = flatten_dos_columns(df_H2)

In [12]:
df_N2.dtypes

Unnamed: 0                                       int64
person                                          object
material                                        object
crystal_struct                                  object
sym_group                                       object
mol                                             object
mol_elecneg                                     object
ad_site_number                                  object
pre_N-N                                        float64
pre_H-H                                        float64
pre_N-H1                                       float64
pre_N-H2                                       float64
pre_N-H3                                       float64
mol_cent_coord                                  object
mol_input_all_coords                            object
n0_atom_type                                    object
n0_atomic_number                                 int64
n0_atomic_mass                                 float64
n0_elect_a

In [13]:
#some columns are added that were not intended, n0_n{}_dos_error are for some datapoints that there was an
#error as a key in the entry dictionary
#lets remove them
df_N2 = df_N2.drop(columns=[col for col in df_N2.columns if re.match(r"n0_n\d+_dos_error", col)])
df_N2H = df_N2H.drop(columns=[col for col in df_N2H.columns if re.match(r"n0_n\d+_dos_error", col)])
df_NH3 = df_NH3.drop(columns=[col for col in df_NH3.columns if re.match(r"n0_n\d+_dos_error", col)])
df_H2 = df_H2.drop(columns=[col for col in df_H2.columns if re.match(r"n0_n\d+_dos_error", col)])

In [14]:
df_N2.dtypes

Unnamed: 0                                       int64
person                                          object
material                                        object
crystal_struct                                  object
sym_group                                       object
mol                                             object
mol_elecneg                                     object
ad_site_number                                  object
pre_N-N                                        float64
pre_H-H                                        float64
pre_N-H1                                       float64
pre_N-H2                                       float64
pre_N-H3                                       float64
mol_cent_coord                                  object
mol_input_all_coords                            object
n0_atom_type                                    object
n0_atomic_number                                 int64
n0_atomic_mass                                 float64
n0_elect_a

In [15]:
#I am trying to convert columns that are mixed object into their literal values
def smart_parse_string_final_safe(value):
    if not isinstance(value, str):
        return value

    stripped = value.strip()

    # ✅ Step 1: Safely parse all valid Python literals
    try:
        return ast.literal_eval(stripped)
    except Exception:
        pass

    # ✅ Step 2: Handle numpy-style `array(...)` values
    if 'array' in stripped:
        try:
            np_safe = re.sub(r'\barray\s*\(', 'np.array(', stripped.replace('\n', ' '))
            return eval(np_safe, {'np': np})
        except Exception:
            return value

    # ✅ Step 3: Malformed space-separated numeric array like "[1.2e-01 3.4e+02]"
    if stripped.startswith("[") and stripped.endswith("]") and ',' not in stripped:
        try:
            number_strs = re.findall(r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?|[-+]?\d+', stripped)
            if number_strs:
                return np.array([float(x) for x in number_strs])
        except Exception:
            pass

    # ❌ Otherwise, return as-is
    return value



# Example DataFrames (replace with yours if loaded again)
dfs = [df_N2, df_N2H, df_NH3, df_H2]
df_names = ['df_N2', 'df_N2H', 'df_NH3', 'df_H2']

# 🚀 Fix all object columns using the final-safe parser
for i, df in enumerate(dfs):
    for col in df.select_dtypes(include='object').columns:
        if df[col].apply(lambda x: isinstance(x, str)).any():
            try:
                df[col] = df[col].apply(smart_parse_string_final_safe)
            except Exception as e:
                print(f"⚠️ {df_names[i]}: Error in column '{col}': {type(e).__name__} - {e}")

# Unpack updated DataFrames
df_N2, df_N2H, df_NH3, df_H2 = dfs



In [16]:
dfs = [df_N2,
      df_H2,
      df_N2H,
      df_NH3]
mol_names = ['N2',
            'H2',
            'N2H',
            'NH3']
print('before')
for i in range(len(dfs_raw)):
    each_df = dfs_raw[i]
    name = mol_names[i]
    print(name,each_df.shape)
print('after')
for i in range(len(dfs)):
    each_df = dfs[i]
    name = mol_names[i]
    print(name,each_df.shape)

before
N2 (489, 637)
H2 (517, 637)
N2H (463, 787)
NH3 (525, 937)
after
N2 (489, 889)
H2 (517, 889)
N2H (463, 1039)
NH3 (525, 1189)


In [18]:
#Lets see if values are converted into their literal values
from collections import Counter
dfs = [df_N2, df_N2H, df_NH3, df_H2]
df_names = ['df_N2', 'df_N2H', 'df_NH3', 'df_H2']
# Dictionary to hold type info for each column
for i in range(0,len(dfs)):
    df = dfs[i]
    non_numeric_types = {}

    for col in df.columns:
      # Get the types in that column (ignoring NaNs)
        types_in_col = [type(val) for val in df[col].dropna()]

      # Count occurrences of each type
        type_counts = Counter(types_in_col)

      # Remove int and float types
        type_counts.pop(int, None)
        type_counts.pop(float, None)

        if type_counts:
            non_numeric_types[col] = type_counts

    #Display columns that have non-int/float entries and their types
    for col, types_found in non_numeric_types.items():
        print(f"{df_names[i]} Column '{col}' has non-numeric types: {dict(types_found)}")

df_N2 Column 'person' has non-numeric types: {<class 'str'>: 489}
df_N2 Column 'material' has non-numeric types: {<class 'str'>: 489}
df_N2 Column 'crystal_struct' has non-numeric types: {<class 'str'>: 489}
df_N2 Column 'sym_group' has non-numeric types: {<class 'str'>: 489}
df_N2 Column 'mol' has non-numeric types: {<class 'str'>: 489}
df_N2 Column 'mol_elecneg' has non-numeric types: {<class 'list'>: 489}
df_N2 Column 'ad_site_number' has non-numeric types: {<class 'str'>: 489}
df_N2 Column 'mol_cent_coord' has non-numeric types: {<class 'list'>: 489}
df_N2 Column 'mol_input_all_coords' has non-numeric types: {<class 'list'>: 489}
df_N2 Column 'n0_atom_type' has non-numeric types: {<class 'str'>: 489}
df_N2 Column 'n0_ion_e' has non-numeric types: {<class 'list'>: 489}
df_N2 Column 'n0_coordinates' has non-numeric types: {<class 'tuple'>: 489}
df_N2 Column 'n0_distance_vector' has non-numeric types: {<class 'tuple'>: 489}
df_N2 Column 'n1_atom_type' has non-numeric types: {<class 's

df_N2H Column 'person' has non-numeric types: {<class 'str'>: 463}
df_N2H Column 'material' has non-numeric types: {<class 'str'>: 463}
df_N2H Column 'crystal_struct' has non-numeric types: {<class 'str'>: 463}
df_N2H Column 'sym_group' has non-numeric types: {<class 'str'>: 463}
df_N2H Column 'mol' has non-numeric types: {<class 'str'>: 463}
df_N2H Column 'mol_elecneg' has non-numeric types: {<class 'list'>: 463}
df_N2H Column 'ad_site_number' has non-numeric types: {<class 'str'>: 463}
df_N2H Column 'mol_cent_coord' has non-numeric types: {<class 'list'>: 463}
df_N2H Column 'mol_input_all_coords' has non-numeric types: {<class 'list'>: 463}
df_N2H Column 'n0_atom_type' has non-numeric types: {<class 'str'>: 463}
df_N2H Column 'n0_ion_e' has non-numeric types: {<class 'list'>: 463}
df_N2H Column 'n0_coordinates' has non-numeric types: {<class 'tuple'>: 463}
df_N2H Column 'n0_distance_vector' has non-numeric types: {<class 'tuple'>: 463}
df_N2H Column 'n1_atom_type' has non-numeric typ

df_NH3 Column 'person' has non-numeric types: {<class 'str'>: 525}
df_NH3 Column 'material' has non-numeric types: {<class 'str'>: 525}
df_NH3 Column 'crystal_struct' has non-numeric types: {<class 'str'>: 525}
df_NH3 Column 'sym_group' has non-numeric types: {<class 'str'>: 525}
df_NH3 Column 'mol' has non-numeric types: {<class 'str'>: 525}
df_NH3 Column 'mol_elecneg' has non-numeric types: {<class 'list'>: 525}
df_NH3 Column 'ad_site_number' has non-numeric types: {<class 'str'>: 525}
df_NH3 Column 'mol_cent_coord' has non-numeric types: {<class 'list'>: 525}
df_NH3 Column 'mol_input_all_coords' has non-numeric types: {<class 'list'>: 525}
df_NH3 Column 'n0_atom_type' has non-numeric types: {<class 'str'>: 525}
df_NH3 Column 'n0_ion_e' has non-numeric types: {<class 'list'>: 525}
df_NH3 Column 'n0_coordinates' has non-numeric types: {<class 'tuple'>: 525}
df_NH3 Column 'n0_distance_vector' has non-numeric types: {<class 'tuple'>: 525}
df_NH3 Column 'n1_atom_type' has non-numeric typ

df_H2 Column 'person' has non-numeric types: {<class 'str'>: 517}
df_H2 Column 'material' has non-numeric types: {<class 'str'>: 517}
df_H2 Column 'crystal_struct' has non-numeric types: {<class 'str'>: 517}
df_H2 Column 'sym_group' has non-numeric types: {<class 'str'>: 517}
df_H2 Column 'mol' has non-numeric types: {<class 'str'>: 517}
df_H2 Column 'mol_elecneg' has non-numeric types: {<class 'list'>: 517}
df_H2 Column 'ad_site_number' has non-numeric types: {<class 'str'>: 517}
df_H2 Column 'mol_cent_coord' has non-numeric types: {<class 'list'>: 517}
df_H2 Column 'mol_input_all_coords' has non-numeric types: {<class 'list'>: 517}
df_H2 Column 'n0_atom_type' has non-numeric types: {<class 'str'>: 517}
df_H2 Column 'n0_ion_e' has non-numeric types: {<class 'list'>: 517}
df_H2 Column 'n0_coordinates' has non-numeric types: {<class 'tuple'>: 517}
df_H2 Column 'n0_distance_vector' has non-numeric types: {<class 'tuple'>: 517}
df_H2 Column 'n1_atom_type' has non-numeric types: {<class 's

In [19]:
df_NH3.loc[20,'n0_distance_vector']

(-0.7118690265800005, 1.0104248121109976, -0.8370944990950004)

In [21]:
##Add a column regarding whether molecule is dissociated, adsorbed molecule and dissociated molecules are
#mutually exclusive
df_H2 = df_H2.copy()
df_N2 = df_N2.copy()
df_N2H = df_N2H.copy()
df_NH3 = df_NH3.copy()

# H2: dissociated if H–H bond is longer than 1.00 Å
df_H2['is_dissociated'] = (df_H2['post_H-H'] > 1.00).astype(int)

# N2: dissociated if N≡N bond is longer than 1.30 Å
df_N2['is_dissociated'] = (df_N2['post_N-N'] > 1.30).astype(int)

# N2H: dissociated if N–N > 1.45 or shortest N–H > 1.25
df_N2H['is_dissociated'] = (
    (df_N2H['post_N-N'] > 1.45) | (df_N2H['post_N-H1'] > 1.25)
).astype(int)

# NH3: dissociated if any N–H bond is longer than 1.25 Å
df_NH3['is_dissociated'] = (
    (df_NH3['post_N-H1'] > 1.25) |
    (df_NH3['post_N-H2'] > 1.25) |
    (df_NH3['post_N-H3'] > 1.25)
).astype(int)

In [23]:
##Define whether adsorption happens in a molecular or atomic way.
df_H2 = df_H2.copy()
df_N2 = df_N2.copy()
df_N2H = df_N2H.copy()
df_NH3 = df_NH3.copy()
# Define adsorption threshold
adsorption_threshold = 2.5

# H2: H0 and H1 distances to metal0
df_H2['Mol/Atom_is_adsorbed'] = (
    (df_H2['H0-metal0_distance'] < adsorption_threshold) |
     (df_H2['H1-metal0_distance'] < adsorption_threshold)
).astype(int)

# N2: N0 and N1 distances to metal0
df_N2['Mol/Atom_is_adsorbed'] = (
    (df_N2['N0-metal0_distance'] < adsorption_threshold) |
     (df_N2['N1-metal0_distance'] < adsorption_threshold)
).astype(int)

# N2H: N0, N1, H0 distances to metal0
df_N2H['Mol/Atom_is_adsorbed'] = ( 
    (df_N2H['N0-metal0_distance'] < adsorption_threshold) |
     (df_N2H['N1-metal0_distance'] < adsorption_threshold) |
     (df_N2H['H0-metal0_distance'] < adsorption_threshold)
).astype(int)

# NH3: N0, H0, H1, H2 distances to metal0
df_NH3['Mol/Atom_is_adsorbed'] = (
     (df_NH3['N0-metal0_distance'] < adsorption_threshold) |
     (df_NH3['H0-metal0_distance'] < adsorption_threshold) |
     (df_NH3['H1-metal0_distance'] < adsorption_threshold) |
     (df_NH3['H2-metal0_distance'] < adsorption_threshold)
).astype(int)




In [24]:
df_N2[(df_N2['Mol/Atom_is_adsorbed']==1) & (df_N2['is_dissociated']==1)][['person','material','ad_site_number','Mol/Atom_is_adsorbed']]

,person,material,ad_site_number,Mol/Atom_is_adsorbed
8,Subhmoy,1_CoPt3,3_ad_site,1
19,Subhmoy,3_Fe3Co,1_ad_site,1
24,Subhmoy,3_Fe3Co,6_ad_site,1
27,Subhmoy,4_FeCo,7_ad_site,1
32,Subhmoy,5_FeNi3,3_ad_site,1
54,Subhmoy,7_NiPt,6_ad_site,1
57,Subhmoy,8_VPt,4_ad_site,1
74,Subhmoy,13_Zn3Cu,4_ad_site,1
79,Subhmoy,14_Zn8Cu5,3_ad_site,1
81,Subhmoy,14_Zn8Cu5,7_ad_site,1


In [25]:
##Define whether adsorption happens in a molecular way.
df_H2['Mol_is_adsorbed'] = (
    (df_H2['is_dissociated'] == 0) &
     (df_H2['Mol/Atom_is_adsorbed'] == 1)
).astype(int)

# N2: N0 and N1 distances to metal0
df_N2['Mol_is_adsorbed'] = (
    (df_N2['is_dissociated'] == 0) &
     (df_N2['Mol/Atom_is_adsorbed'] == 1)
).astype(int)

# N2H: N0, N1, H0 distances to metal0
df_N2H['Mol_is_adsorbed'] = (
    (df_N2H['is_dissociated'] == 0) &
     (df_N2H['Mol/Atom_is_adsorbed'] == 1)
).astype(int)

# NH3: N0, H0, H1, H2 distances to metal0
df_NH3['Mol_is_adsorbed'] = (
    (df_NH3['is_dissociated'] == 0) &
     (df_NH3['Mol/Atom_is_adsorbed'] == 1)
).astype(int)

In [26]:
##Define whether adsorption happens in an atomic way.
df_H2['atomic_is_adsorbed'] = (
    (df_H2['is_dissociated'] == 1) &
     (df_H2['Mol/Atom_is_adsorbed'] == 1)
).astype(int)

# N2: N0 and N1 distances to metal0
df_N2['atomic_is_adsorbed'] = (
    (df_N2['is_dissociated'] == 1) &
     (df_N2['Mol/Atom_is_adsorbed'] == 1)
).astype(int)

# N2H: N0, N1, H0 distances to metal0
df_N2H['atomic_is_adsorbed'] = (
    (df_N2H['is_dissociated'] == 1) &
     (df_N2H['Mol/Atom_is_adsorbed'] == 1)
).astype(int)

# NH3: N0, H0, H1, H2 distances to metal0
df_NH3['atomic_is_adsorbed'] = (
    (df_NH3['is_dissociated'] == 1) &
     (df_NH3['Mol/Atom_is_adsorbed'] == 1)
).astype(int)

In [27]:
cols_targeted = ['atomic_is_adsorbed','Mol_is_adsorbed']
dfs = [df_N2, df_N2H, df_NH3, df_H2]
df_names = ["df_N2", "df_N2H", "df_NH3","df_H2"]

for i in range(len(dfs)):
    df = dfs[i]
    name = df_names[i]
    for col in cols_targeted:
        print(name, col, df[col].sum())

df_N2 atomic_is_adsorbed 91
df_N2 Mol_is_adsorbed 255
df_N2H atomic_is_adsorbed 171
df_N2H Mol_is_adsorbed 292
df_NH3 atomic_is_adsorbed 78
df_NH3 Mol_is_adsorbed 377
df_H2 atomic_is_adsorbed 369
df_H2 Mol_is_adsorbed 43


In [29]:
#I want to make sure that molecules are adsorbed to the surface from the N side not the H side
df_H2 = df_H2.copy()
df_N2 = df_N2.copy()
df_N2H = df_N2H.copy()
df_NH3 = df_NH3.copy()
# Helper function to get atom type only ('N' or 'H')
def get_closest_atom_type(row, columns):
    distances = {col: row[col] for col in columns}
    closest_atom = min(distances, key=distances.get)
    return closest_atom[0]  # First character is 'N' or 'H'

# ---- H2 Dataset ----
cols_H2 = ['H0-metal0_distance', 'H1-metal0_distance']
df_H2['closest_atom_to_surface'] = np.where(
    df_H2['Mol_is_adsorbed'] == 1,
    df_H2.apply(lambda row: get_closest_atom_type(row, cols_H2), axis=1),
    None
)

# ---- N2 Dataset ----
cols_N2 = ['N0-metal0_distance', 'N1-metal0_distance']
df_N2['closest_atom_to_surface'] = np.where(
    df_N2['Mol_is_adsorbed'] == 1,
    df_N2.apply(lambda row: get_closest_atom_type(row, cols_N2), axis=1),
    None
)

# ---- N2H Dataset ----
cols_N2H = ['N0-metal0_distance', 'N1-metal0_distance', 'H0-metal0_distance']
df_N2H['closest_atom_to_surface'] = np.where(
    df_N2H['Mol_is_adsorbed'] == 1,
    df_N2H.apply(lambda row: get_closest_atom_type(row, cols_N2H), axis=1),
    None
)

# ---- NH3 Dataset ----
cols_NH3 = ['N0-metal0_distance', 'H0-metal0_distance', 'H1-metal0_distance', 'H2-metal0_distance']
df_NH3['closest_atom_to_surface'] = np.where(
    df_NH3['Mol_is_adsorbed'] == 1,
    df_NH3.apply(lambda row: get_closest_atom_type(row, cols_NH3), axis=1),
    None
)


In [30]:
count_by_atom_H2 = df_H2['closest_atom_to_surface'].value_counts(dropna=True)
count_by_atom_N2 = df_N2['closest_atom_to_surface'].value_counts(dropna=True)
count_by_atom_N2H = df_N2H['closest_atom_to_surface'].value_counts(dropna=True)
count_by_atom_NH3 = df_NH3['closest_atom_to_surface'].value_counts(dropna=True)

# Display the results
print("H2:")
print(count_by_atom_H2)
print("\nN2:")
print(count_by_atom_N2)
print("\nN2H:")
print(count_by_atom_N2H)
print("\nNH3:")
print(count_by_atom_NH3)



H2:
H    43
Name: closest_atom_to_surface, dtype: int64

N2:
N    255
Name: closest_atom_to_surface, dtype: int64

N2H:
N    292
Name: closest_atom_to_surface, dtype: int64

NH3:
N    375
H      2
Name: closest_atom_to_surface, dtype: int64


In [31]:
#Remove the ones in NH3 that are adsorbed from H side.
df_NH3 = df_NH3[df_NH3['closest_atom_to_surface'] != 'H']
count_by_atom_NH3 = df_NH3['closest_atom_to_surface'].value_counts(dropna=True)

print("\nNH3:")
print(count_by_atom_NH3)



NH3:
N    375
Name: closest_atom_to_surface, dtype: int64


In [32]:
## N2H and N2 horizontal or vertical adsorption
import numpy as np

def classify_molecular_orientation(mol_coords, atom_pair=['N', 'N'], vertical_thresh=20, horizontal_thresh=20):
    """
    Computes the angle between a bond vector (defined by atom_pair) and the surface (z-axis),
    and classifies the orientation as vertical, horizontal, or tilted.

    Args:
        mol_coords (list): List of (atom_type, np.array([x, y, z])) tuples.
        atom_pair (list): List of two atom types to define the bond (e.g., ['N', 'N'] or ['N', 'H']).
        vertical_thresh (float): Threshold in degrees for classifying "vertical".
        horizontal_thresh (float): Threshold in degrees for classifying "horizontal".

    Returns:
        dict: {
            'angle': angle in degrees,
            'orientation': 'vertical', 'horizontal', or 'tilted'
        }
    """
    # Validate input
    if len(atom_pair) != 2:
        raise ValueError("atom_pair must be a list of two atom types (e.g., ['N', 'H']).")

    # Find atoms matching the pair (first occurrence only)
    atom1 = atom2 = None
    for atom_type, coord in mol_coords:
        if atom1 is None and atom_type == atom_pair[0]:
            atom1 = np.array(coord)
        elif atom2 is None and atom_type == atom_pair[1]:
            atom2 = np.array(coord)
        if atom1 is not None and atom2 is not None:
            break

    if atom1 is None or atom2 is None:
        raise ValueError(f"Could not find a complete pair {atom_pair} in mol_coords.")

    # Bond vector and normalization
    bond_vector = atom2 - atom1
    bond_vector /= np.linalg.norm(bond_vector)

    # Surface normal (z-axis)
    z_axis = np.array([0, 0, 1])

    # Compute angle in degrees
    angle_rad = np.arccos(np.clip(np.dot(bond_vector, z_axis), -1.0, 1.0))
    angle_deg = np.degrees(angle_rad)

    # Classify orientation
    if angle_deg <= vertical_thresh or abs(angle_deg - 180) <= vertical_thresh:
        orientation = 'vertical'
    elif abs(angle_deg - 90) <= horizontal_thresh:
        orientation = 'horizontal'
    else:
        orientation = 'tilted'

    return {
        'angle': angle_deg,
        'orientation': orientation
    }

# Apply only to adsorbed molecules
df_N2.loc[df_N2['Mol_is_adsorbed'] == 1, ['angle', 'orientation']] = (
    df_N2.loc[df_N2['Mol_is_adsorbed'] == 1, 'mol_output_all_coords']
    .apply(lambda coords: pd.Series(classify_molecular_orientation(coords, atom_pair=['N', 'N'])))
)

df_N2H.loc[df_N2H['Mol_is_adsorbed'] == 1, ['angle', 'orientation']] = (
    df_N2H.loc[df_N2H['Mol_is_adsorbed'] == 1, 'mol_output_all_coords']
    .apply(lambda coords: pd.Series(classify_molecular_orientation(coords, atom_pair=['N', 'N'])))
)

In [35]:
dfs = [df_N2,
      df_H2,
      df_N2H,
      df_NH3]
mol_names = ['N2',
            'H2',
            'N2H',
            'NH3']

for i in range(len(dfs)):
    df = dfs[i]
    name = mol_names[i]
    cols = [col for col in df.columns if 'index' not in col.lower()]
    dfs[i] = df[cols]
    
df_N2, df_H2, df_N2H, df_NH3 = dfs

In [36]:
df_N2.to_csv('./dataframes/roughly_cleaned_bandcenter_df_N2_bandcenter_v3.csv', index=False)
df_N2H.to_csv('./dataframes/roughly_cleaned_bandcenter_df_N2H_bandcenter_v3.csv', index=False)
df_NH3.to_csv('./dataframes/roughly_cleaned_bandcenter_df_NH3_bandcenter_v3.csv', index=False)
df_H2.to_csv('./dataframes/roughly_cleaned_bandcenter_df_H2_bandcenter_v3.csv', index=False)

In [3]:
df_N2 = pd.read_csv('./dataframes/roughly_cleaned_bandcenter_df_N2_bandcenter_v3.csv')
df_N2H = pd.read_csv('./dataframes/roughly_cleaned_bandcenter_df_N2H_bandcenter_v3.csv')
df_NH3 = pd.read_csv('./dataframes/roughly_cleaned_bandcenter_df_NH3_bandcenter_v3.csv')
df_H2 = pd.read_csv('./dataframes/roughly_cleaned_bandcenter_df_H2_bandcenter_v3.csv')

In [4]:
# 1️⃣ Define the shared initial and final columns (for df_N2)
initial_cols = [
    'material',
    'ad_site_number',
    'crystal_struct', 'sym_group', 'mol', 'mol_elecneg',
    'pre_N-N', 'pre_H-H', 'pre_N-H1', 'pre_N-H2', 'pre_N-H3',
    'mol_cent_coord', 'mol_input_all_coords'
]
ending_cols = [
    'mol_output_all_coords','is_dissociated', 'Mol_is_adsorbed', 'atomic_is_adsorbed',
    'angle', 'orientation', 'ads_e'
]

# 2️⃣ Define input-specific middle columns (n0–n20 + dos)
input_middle_cols = []
for i in range(21):
    input_middle_cols += [col for col in df_N2.columns if col.startswith(f'n{i}_')]


# 3️⃣ Safe filtering to ensure columns exist
input_cols = [col for col in initial_cols + input_middle_cols + ending_cols if col in df_N2.columns]

# 4️⃣ Create input DataFrame
input_df_N2 = df_N2[input_cols]

# 5️⃣ Define output-specific middle columns
output_middle_cols = [
    'post_N-N', 'post_H-H', 'post_N-H1', 'post_N-H2', 'post_N-H3',
]
# add N0-metal0...N0-metal9 and N1-metal0...N1-metal9 sets
for prefix in ['N0-metal', 'N1-metal']:
    for j in range(10):
        suffixes = [
            '_closest_surface_atom_type', '_atomic_number', '_atomic_mass',
            '_elect_affinity', '_ion_e', '_atomic_radius', '_cov_rad',
            '_atom_density', '_closest_surface_atom_coords',
            '_closest_surface_atom_bader_charge',
            '_corrected_bader_charge',
            '_closest_surface_atom_elneg',
            '_distance', '_distance_vector'
        ]
        for s in suffixes:
            output_middle_cols.append(f"{prefix}{j}{s}")

# 6️⃣ Safe filtering
output_cols = [col for col in initial_cols + output_middle_cols + ending_cols if col in df_N2.columns]

# 7️⃣ Create output DataFrame
output_df_N2 = df_N2[output_cols]


In [5]:
# 1️⃣ Shared initial and final columns (adjusted for df_H2)
initial_cols = [
    'material',
    'ad_site_number',
    'crystal_struct', 'sym_group', 'mol', 'mol_elecneg',
    'pre_N-N', 'pre_H-H', 'pre_N-H1', 'pre_N-H2', 'pre_N-H3',
    'mol_cent_coord', 'mol_input_all_coords'
]
ending_cols = [
    'mol_output_all_coords','is_dissociated', 'Mol_is_adsorbed', 'atomic_is_adsorbed', 'ads_e'
]

# 2️⃣ Input middle columns (n0-n20 + future dos support)
input_middle_cols = []
for i in range(21):
    input_middle_cols += [col for col in df_H2.columns if col.startswith(f'n{i}_')]


# 3️⃣ Construct final input column list and DataFrame
input_cols = [col for col in initial_cols + input_middle_cols + ending_cols if col in df_H2.columns]
input_df_H2 = df_H2[input_cols]

# 4️⃣ Output middle columns (post_ and H0/H1 metal descriptors)
output_middle_cols = ['post_N-N', 'post_H-H', 'post_N-H1', 'post_N-H2', 'post_N-H3']

# Add H0-metal0 to H0-metal9 and H1-metal0 to H1-metal9
for prefix in ['H0-metal', 'H1-metal']:
    for j in range(10):
        suffixes = [
            '_closest_surface_atom_type', '_atomic_number', '_atomic_mass',
            '_elect_affinity', '_ion_e', '_atomic_radius', '_cov_rad',
            '_atom_density', '_closest_surface_atom_coords',
            '_closest_surface_atom_bader_charge',
            '_corrected_bader_charge',
            '_closest_surface_atom_elneg',
            '_distance', '_distance_vector'
        ]
        for s in suffixes:
            output_middle_cols.append(f"{prefix}{j}{s}")

# 5️⃣ Final output DataFrame
output_cols = [col for col in initial_cols + output_middle_cols + ending_cols if col in df_H2.columns]
output_df_H2 = df_H2[output_cols]


In [6]:
# 1️⃣ Shared initial and ending columns (for df_N2H)
initial_cols = [
    'material',
    'ad_site_number',
    'crystal_struct', 'sym_group', 'mol', 'mol_elecneg',
    'pre_N-N', 'pre_H-H', 'pre_N-H1', 'pre_N-H2', 'pre_N-H3',
    'mol_cent_coord', 'mol_input_all_coords'
]

ending_cols = [
    'mol_output_all_coords','is_dissociated', 'Mol_is_adsorbed', 'atomic_is_adsorbed',
    'angle', 'orientation', 'ads_e'
]

# 2️⃣ Input middle: n0–n20 sets plus any n0_nX_dos
input_middle_cols = []
for i in range(21):
    input_middle_cols += [
        col for col in df_N2H.columns if col.startswith(f'n{i}_')
    ]


# 3️⃣ Build input DataFrame
input_cols = [
    col for col in initial_cols + input_middle_cols + ending_cols
    if col in df_N2H.columns
]
input_df_N2H = df_N2H[input_cols]

# 4️⃣ Output middle: bonds (post-…) + N0-metal/N1-metal series
output_middle_cols = [
    'post_N-N', 'post_H-H', 'post_N-H1',
    'post_N-H2', 'post_N-H3'
]

for prefix in ['N0-metal', 'N1-metal','H0-metal']:
    for j in range(10):
        suffixes = [
            '_closest_surface_atom_type', '_atomic_number',
            '_atomic_mass', '_elect_affinity', '_ion_e',
            '_atomic_radius', '_cov_rad', '_atom_density',
            '_closest_surface_atom_coords',
            '_closest_surface_atom_bader_charge',
            '_corrected_bader_charge',
            '_closest_surface_atom_elneg', '_distance',
            '_distance_vector'
        ]
        output_middle_cols += [
            f"{prefix}{j}{s}" for s in suffixes
        ]

# 5️⃣ Build output DataFrame
output_cols = [
    col for col in initial_cols + output_middle_cols + ending_cols
    if col in df_N2H.columns
]
output_df_N2H = df_N2H[output_cols]


In [7]:
# 1️⃣ Shared initial and final columns (adjusted for df_NH3)
initial_cols = [
    'material',
    'ad_site_number',
    'crystal_struct', 'sym_group', 'mol', 'mol_elecneg',
    'pre_N-N', 'pre_H-H', 'pre_N-H1', 'pre_N-H2', 'pre_N-H3',
    'mol_cent_coord', 'mol_input_all_coords'
]
ending_cols = [
    'mol_output_all_coords','is_dissociated', 'Mol_is_adsorbed', 'atomic_is_adsorbed', 'ads_e'
]

# 2️⃣ Input middle columns (n0-n20 + future dos support)
input_middle_cols = []
for i in range(21):
    input_middle_cols += [col for col in df_NH3.columns if col.startswith(f'n{i}_')]

# 3️⃣ Construct final input column list and DataFrame
input_cols = [col for col in initial_cols + input_middle_cols + ending_cols if col in df_NH3.columns]
input_df_NH3 = df_NH3[input_cols]

# 4️⃣ Output middle columns (post_ and H0/H1 metal descriptors)
output_middle_cols = ['post_N-N', 'post_H-H', 'post_N-H1', 'post_N-H2', 'post_N-H3']

# Add H0-metal0 to H0-metal9 and H1-metal0 to H1-metal9
for prefix in ['N0-metal','H0-metal', 'H1-metal','H2-metal']:
    for j in range(10):
        suffixes = [
            '_closest_surface_atom_type', '_atomic_number', '_atomic_mass',
            '_elect_affinity', '_ion_e', '_atomic_radius', '_cov_rad',
            '_atom_density', '_closest_surface_atom_coords',
            '_closest_surface_atom_bader_charge',
            '_corrected_bader_charge',
            '_closest_surface_atom_elneg',
            '_distance', '_distance_vector'
        ]
        for s in suffixes:
            output_middle_cols.append(f"{prefix}{j}{s}")

# 5️⃣ Final output DataFrame
output_cols = [col for col in initial_cols + output_middle_cols + ending_cols if col in df_NH3.columns]
output_df_NH3 = df_NH3[output_cols]


In [8]:
input_df_N2.to_csv('./dataframes/input_df_N2_bandcenter_v1.csv', index=False)
input_df_N2H.to_csv('./dataframes/input_df_N2H_bandcenter_v1.csv', index=False)
input_df_NH3.to_csv('./dataframes/input_df_NH3_bandcenter_v1.csv', index=False)
input_df_H2.to_csv('./dataframes/input_df_H2_bandcenter_v1.csv', index=False)

output_df_N2.to_csv('./dataframes/output_df_N2_bandcenter_v1.csv', index=False)
output_df_N2H.to_csv('./dataframes/output_df_N2H_bandcenter_v1.csv', index=False)
output_df_NH3.to_csv('./dataframes/output_df_NH3_bandcenter_v1.csv', index=False)
output_df_H2.to_csv('./dataframes/output_df_H2_bandcenter_v1.csv', index=False)

In [9]:
#This compression is somehow useless but if you want to use it go ahead
input_df_N2 = pd.read_csv('./dataframes/input_df_N2.csv')
input_df_N2H= pd.read_csv('./dataframes/input_df_N2H.csv')
input_df_NH3= pd.read_csv('./dataframes/input_df_NH3.csv')
input_df_H2=  pd.read_csv('./dataframes/input_df_H2.csv')

In [45]:
#Group nrighbor atoms properties
def compress_neighbors(df,n_neighbor):
    df = df.copy()
    neighbor_indices = range(n_neighbor)
    properties = [
        'atom_type', 'atomic_number', 'atomic_mass', 'elect_affinity', 'ion_e',
        'atomic_radius', 'cov_rad', 'atom_density', 'coordinates',
        'distance_vector', 'distance', 'charge', 'electroneg'
    ]

    # Initial core features to keep at the beginning
    initial_cols = [
        'crystal_struct', 'sym_group', 'mol', 'mol_elecneg',
        'pre_N-N', 'pre_H-H', 'pre_N-H1', 'pre_N-H2', 'pre_N-H3',
        'mol_cent_coord', 'mol_input_all_coords'
    ]
    
    # Safe check: only include those that exist in this df
    initial_cols = [col for col in initial_cols if col in df.columns]

    # Identify dos columns (if they exist)
    dos_cols = [
        col for col in df.columns
        if col.startswith('n0_n') and col.endswith('_dos')
    ]

    # Create grouped dict columns for neighbors
    for i in neighbor_indices:
        prop_cols = [f'n{i}_{p}' for p in properties if f'n{i}_{p}' in df.columns]
        if prop_cols:
            df[f'n{i}'] = df[prop_cols].apply(lambda row: {
                p: row[f'n{i}_{p}'] for p in properties if f'n{i}_{p}' in df.columns
            }, axis=1)

    # Drop original neighbor property columns, exclude dos
    drop_cols = [
        col for col in df.columns
        if any(col.startswith(f'n{i}_') for i in neighbor_indices)
        and col not in dos_cols
    ]
    df.drop(columns=drop_cols, inplace=True)

    # Final ordering
    neighbor_dict_cols = [f'n{i}' for i in neighbor_indices if f'n{i}' in df.columns]
    middle_cols = neighbor_dict_cols + dos_cols
    final_cols = [col for col in df.columns if col not in initial_cols + middle_cols]

    ordered_cols = initial_cols + middle_cols + final_cols
    return df[ordered_cols]


In [46]:
compressed_input_df_N2 = compress_neighbors(input_df_N2,21)
compressed_input_df_N2H = compress_neighbors(input_df_N2H,21)
compressed_input_df_NH3 = compress_neighbors(input_df_NH3,21)
compressed_input_df_H2 = compress_neighbors(input_df_H2,21)

In [48]:
compressed_input_df_H2.columns.tolist()

['crystal_struct',
 'sym_group',
 'mol',
 'mol_elecneg',
 'pre_N-N',
 'pre_H-H',
 'pre_N-H1',
 'pre_N-H2',
 'pre_N-H3',
 'mol_cent_coord',
 'mol_input_all_coords',
 'n0',
 'n1',
 'n2',
 'n3',
 'n4',
 'n5',
 'n6',
 'n7',
 'n8',
 'n9',
 'n10',
 'n11',
 'n12',
 'n13',
 'n14',
 'n15',
 'n16',
 'n17',
 'n18',
 'n19',
 'n20',
 'n0_n0_dos',
 'n0_n1_dos',
 'n0_n2_dos',
 'n0_n3_dos',
 'n0_n4_dos',
 'n0_n5_dos',
 'n0_n6_dos',
 'n0_n7_dos',
 'n0_n8_dos',
 'n0_n9_dos',
 'n0_n10_dos',
 'n0_n11_dos',
 'n0_n12_dos',
 'n0_n13_dos',
 'n0_n14_dos',
 'n0_n15_dos',
 'n0_n16_dos',
 'n0_n17_dos',
 'n0_n18_dos',
 'n0_n19_dos',
 'n0_n20_dos',
 'mol_output_all_coords',
 'is_dissociated',
 'Mol_is_adsorbed',
 'atomic_is_adsorbed',
 'ads_e']